# CMT316 - VOC Localisation (Faster R-CNN Pipeline)

This notebook follows an end-to-end Computer Vision Machine Learning pipeline for VOC object localisation using Faster R-CNN.

## Pipeline Overview
1. Requirements and runtime setup.
2. Dataset bootstrap and split integrity checks.
3. Shared VOC parsing + EDA diagnostics.
4. Model setup, training modes, and experiment execution.
5. Validation-driven model selection.
6. Export predictions and report-ready evaluation outputs.

## Important evaluation note
- Full multi-class VOC evaluation is performed on the authors' official evaluation server for final submission.
- Inside this notebook, detailed local evaluation and visualisation focus on the `person` class only (as required for local reporting).

## Table of Contents
- [1. Environment and Requirements](#1-environment-and-requirements)
- [2. Dataset Organisation and Bootstrap](#2-dataset-organisation-and-bootstrap)
- [3. Shared VOC Data Construction](#3-shared-voc-data-construction)
- [4. Exploratory Data Analysis](#4-exploratory-data-analysis)
- [5. Faster R-CNN Setup and Training Utilities](#5-faster-r-cnn-setup-and-training-utilities)
- [6. Experiment Suite (Five Explicit Runs)](#6-experiment-suite-five-explicit-runs)
- [7. Evaluation and Visualisation (Person Class)](#7-evaluation-and-visualisation-person-class)
- [8. Predict and Export](#8-predict-and-export)

In [7]:
from pathlib import Path
import re, shutil, zipfile, os, gdown

def get_project_root() -> Path:
    curr = Path.cwd()
    for parent in [curr] + list(curr.parents):
        if (parent / 'version_2').exists() or (parent / '.git').exists():
            return parent
    return curr

PROJECT_ROOT = get_project_root()
DATA_ROOT = PROJECT_ROOT / 'version_2' / 'data'
BOOTSTRAP_DATA_ROOT = PROJECT_ROOT / 'voc_dataset'
ARTIFACTS_DIR = PROJECT_ROOT / 'artifacts'
ARCHIVE_DIR = PROJECT_ROOT / 'archives'
ARCHIVE_DIR.mkdir(parents=True, exist_ok=True)

DATASET_LINK = 'https://drive.google.com/file/d/1M8EPREjTLAqY3VAjfruDcxtaudXy4xm2/view?usp=share_link'
FORCE_REDOWNLOAD, FORCE_REEXTRACT = False, False

def extract_drive_file_id(link):
    match = re.search(r'/file/d/([a-zA-Z0-9_-]+)', link)
    return match.group(1) if match else None

def ensure_archive(link, out_zip):
    if out_zip.exists(): return out_zip
    fid = extract_drive_file_id(link)
    gdown.download(f'https://drive.google.com/uc?id={fid}', str(out_zip), quiet=False)
    return out_zip

def install_voc(split_name, arch_path, root):
    target = root / split_name
    if target.exists(): return target
    tmp = root / '_tmp' / split_name
    tmp.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(arch_path, 'r') as zf: zf.extractall(tmp)
    shutil.move(str(tmp), str(target))
    return target

dataset_zip = ensure_archive(DATASET_LINK, ARCHIVE_DIR / 'dataset.zip')
voc_dataset_root = install_voc('voc_dataset', dataset_zip, PROJECT_ROOT)
print('Project Root:', PROJECT_ROOT)


Project Root: /Users/michaelkaramichalis/Library/Mobile Documents/com~apple~CloudDocs/MSc Advanced Computer Science/CMT316 - Applications of Machine Learning/Assessment 2/VOC_MAIN


## 2. Dataset Organisation and Bootstrap

This notebook uses the already materialized versioned VOC split stored under `version_2/data/`:

- `version_2/data/clean_ids/{train,val,dev_test,test}.txt`
- `version_2/data/images/{train,val,dev_test,test}/`
- `version_2/data/annotations/{train,val,dev_test,test}/`

Run this section to verify the local layout before training. No download or extraction is performed here.

In [8]:
from pathlib import Path

requirements_path = Path('requirements.txt')

if requirements_path.exists():
    %pip install -r requirements.txt
else:
    %pip install numpy pandas matplotlib seaborn pillow torch torchvision transformers accelerate ultralytics


[notice] A new release of pip is available: 26.0 -> 26.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


### Legacy bootstrap note

The archive-based bootstrap used in earlier drafts has been removed. The notebook now uses the versioned split layout under `version_2/data/` throughout.

In [9]:
from pathlib import Path

# Using unified BOOTSTRAP_DATA_ROOT from Section 1
SPLIT_DIR = BOOTSTRAP_DATA_ROOT / 'clean_ids'
IMAGE_ROOT = BOOTSTRAP_DATA_ROOT / 'images'
ANNOTATION_ROOT = BOOTSTRAP_DATA_ROOT / 'annotations'
EXPECTED_SPLITS = ['train', 'val', 'dev_test', 'test']

missing_paths = [path for path in [SPLIT_DIR, IMAGE_ROOT, ANNOTATION_ROOT] if not path.exists()]
if missing_paths:
    print(f'Warning: Some expected paths are missing in {BOOTSTRAP_DATA_ROOT}')

print('Bootstrap Dataset root:', BOOTSTRAP_DATA_ROOT)
if SPLIT_DIR.exists():
    print('Split files:', sorted(path.name for path in SPLIT_DIR.glob('*.txt')))


Bootstrap Dataset root: /Users/michaelkaramichalis/Library/Mobile Documents/com~apple~CloudDocs/MSc Advanced Computer Science/CMT316 - Applications of Machine Learning/Assessment 2/VOC_MAIN/voc_dataset
Split files: ['dev_test.txt', 'split_seed.txt', 'test.txt', 'train.txt', 'val.txt']


## 3. Shared VOC Data Construction

This stage initialises the runtime, dataset paths, class dictionaries, and split-aware parsing utilities.

The implementation prioritises split hygiene:
- train/val leakage is prevented
- test remains untouched for final reporting
- IDs are robustly parsed from both one-column and two-column VOC list formats

In [10]:
from pathlib import Path
import random
import xml.etree.ElementTree as ET

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from PIL import Image
from torch.utils.data import DataLoader, Dataset

sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 200)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', DEVICE)
print('Seed:', SEED)

Using device: cpu
Seed: 42


In [11]:
# Using unified V2_DATA_ROOT and ARTIFACTS_DIR from Section 1
SPLIT_ID_DIR = V2_DATA_ROOT / 'clean_ids'
IMAGE_ROOT = V2_DATA_ROOT / 'images'
ANNOTATION_ROOT = V2_DATA_ROOT / 'annotations'
TRAIN_IMAGE_DIR = IMAGE_ROOT / 'train'
VAL_IMAGE_DIR = IMAGE_ROOT / 'val'
DEV_TEST_IMAGE_DIR = IMAGE_ROOT / 'dev_test'
TEST_IMAGE_DIR = IMAGE_ROOT / 'test'
TRAIN_ANN_DIR = ANNOTATION_ROOT / 'train'
VAL_ANN_DIR = ANNOTATION_ROOT / 'val'
DEV_TEST_ANN_DIR = ANNOTATION_ROOT / 'dev_test'
TEST_ANN_DIR = ANNOTATION_ROOT / 'test'
GROUP_SPLIT_DIR = ARTIFACTS_DIR / 'group_split'
PREDICTIONS_DIR = ARTIFACTS_DIR / 'predictions'
PREDICTIONS_DIR.mkdir(parents=True, exist_ok=True)
EXPERIMENT_RUNS_DIR = ARTIFACTS_DIR / 'experiment_runs'
EXPERIMENT_RUNS_DIR.mkdir(parents=True, exist_ok=True)

VOC_CLASSES = [
    'aeroplane', 'bicycle', 'bird', 'boat', 'bottle',
    'bus', 'car', 'cat', 'chair', 'cow',
    'diningtable', 'dog', 'horse', 'motorbike', 'person',
    'pottedplant', 'sheep', 'sofa', 'train', 'tvmonitor',
]
CLASS_TO_ID = {name: idx for idx, name in enumerate(VOC_CLASSES)}
ID_TO_CLASS = {idx: name for name, idx in CLASS_TO_ID.items()}

PREDICTION_COLUMNS = [
    'image_id', 'split', 'source_model', 'class_id', 'cls',
    'xmin', 'ymin', 'xmax', 'ymax', 'score'
]

for path in [TRAIN_IMAGE_DIR, VAL_IMAGE_DIR, DEV_TEST_IMAGE_DIR, TEST_IMAGE_DIR, TRAIN_ANN_DIR, VAL_ANN_DIR, DEV_TEST_ANN_DIR, TEST_ANN_DIR]:
    if not path.exists():
        raise FileNotFoundError(f'Missing expected VOC folder: {path}')

print('PROJECT_ROOT:', PROJECT_ROOT)
print('DATA_ROOT:', DATA_ROOT)
print('SPLIT_ID_DIR:', SPLIT_ID_DIR)
print('TRAIN_IMAGE_DIR:', TRAIN_IMAGE_DIR)
print('VAL_IMAGE_DIR:', VAL_IMAGE_DIR)
print('TEST_IMAGE_DIR:', TEST_IMAGE_DIR)

NameError: name 'V2_DATA_ROOT' is not defined

### 3.1 Split Parsing and Annotation Tables

This section builds image-level and object-level tables from VOC XML files and enforces clean split boundaries.

Outputs produced here are reused by both EDA and model training.

In [ ]:
def read_ids_from_file(path: Path) -> list[str]:
    if not path.exists():
        return []

    # Handles both formats:
    # 1) "image_id"
    # 2) "image_id <object_index>" (VOC Layout lists)
    ids = []
    seen = set()
    for line in path.read_text().splitlines():
        tokens = line.strip().split()
        if not tokens:
            continue
        image_id = tokens[0]
        if image_id not in seen:
            ids.append(image_id)
            seen.add(image_id)
    return ids



def load_split_ids(split_name: str) -> list[str]:
    # Split IDs are materialized under version_2/data/clean_ids.
    split_path = SPLIT_ID_DIR / f'{split_name}.txt'
    split_ids = read_ids_from_file(split_path)
    if not split_ids:
        raise FileNotFoundError(f'Missing split file: {split_path}')
    return split_ids



def parse_annotation(xml_path: Path, image_id: str, split_name: str) -> tuple[dict, list[dict]]:
    root = ET.parse(xml_path).getroot()

    size_node = root.find('size')
    width = int(size_node.findtext('width', default='0')) if size_node is not None else 0
    height = int(size_node.findtext('height', default='0')) if size_node is not None else 0

    records = []
    for obj in root.findall('object'):
        cls = obj.findtext('name', default='').strip()
        if cls not in CLASS_TO_ID:
            continue

        box = obj.find('bndbox')
        if box is None:
            continue

        xmin = int(float(box.findtext('xmin', default='0')))
        ymin = int(float(box.findtext('ymin', default='0')))
        xmax = int(float(box.findtext('xmax', default='0')))
        ymax = int(float(box.findtext('ymax', default='0')))

        box_w = max(1, xmax - xmin + 1)
        box_h = max(1, ymax - ymin + 1)

        records.append({
            'image_id': image_id,
            'split': split_name,
            'cls': cls,
            'class_id': CLASS_TO_ID[cls],
            'xmin': xmin,
            'ymin': ymin,
            'xmax': xmax,
            'ymax': ymax,
            'box_w': box_w,
            'box_h': box_h,
            'box_area': box_w * box_h,
            'difficult': int(obj.findtext('difficult', default='0')),
            'truncated': int(obj.findtext('truncated', default='0')),
            'occluded': int(obj.findtext('occluded', default='0')),
            'width': width,
            'height': height,
        })

    image_meta = {
        'image_id': image_id,
        'split': split_name,
        'width': width,
        'height': height,
    }
    return image_meta, records



def build_split_tables(split_name: str, image_ids: list[str]) -> tuple[pd.DataFrame, pd.DataFrame]:
    image_rows, object_rows = [], []
    ann_dir = ANNOTATION_ROOT / split_name

    for image_id in image_ids:
        xml_path = ann_dir / f'{image_id}.xml'
        if not xml_path.exists():
            image_rows.append({'image_id': image_id, 'split': split_name, 'width': np.nan, 'height': np.nan})
            continue

        image_meta, rows = parse_annotation(xml_path, image_id, split_name)
        image_rows.append(image_meta)
        object_rows.extend(rows)

    images_df = pd.DataFrame(image_rows).drop_duplicates(subset=['image_id', 'split'])
    objects_df = pd.DataFrame(object_rows)

    if not objects_df.empty:
        objects_df['box_w'] = (objects_df['xmax'] - objects_df['xmin'] + 1).clip(lower=1)
        objects_df['box_h'] = (objects_df['ymax'] - objects_df['ymin'] + 1).clip(lower=1)
        objects_df['box_area'] = objects_df['box_w'] * objects_df['box_h']
        objects_df['aspect_ratio'] = objects_df['box_w'] / objects_df['box_h']
        image_area = (objects_df['width'] * objects_df['height']).replace(0, np.nan)
        objects_df['norm_area'] = objects_df['box_area'] / image_area

    return images_df, objects_df


train_ids = load_split_ids('train')
val_ids = load_split_ids('val')
dev_test_ids = load_split_ids('dev_test')
test_ids = load_split_ids('test')

train_images, train_objects = build_split_tables('train', train_ids)
val_images, val_objects = build_split_tables('val', val_ids)
dev_test_images, dev_test_objects = build_split_tables('dev_test', dev_test_ids)
test_images, test_objects = build_split_tables('test', test_ids)

train_model_df = train_objects[(train_objects['xmax'] >= train_objects['xmin']) & (train_objects['ymax'] >= train_objects['ymin'])].copy()
val_model_df = val_objects[(val_objects['xmax'] >= val_objects['xmin']) & (val_objects['ymax'] >= val_objects['ymin'])].copy()
dev_test_model_df = dev_test_objects[(dev_test_objects['xmax'] >= dev_test_objects['xmin']) & (dev_test_objects['ymax'] >= dev_test_objects['ymin'])].copy()

train_model_df_ignoring_difficult = train_model_df[train_model_df['difficult'] == 0].reset_index(drop=True)
val_model_df_ignoring_difficult = val_model_df[val_model_df['difficult'] == 0].reset_index(drop=True)
dev_test_model_df_ignoring_difficult = dev_test_model_df[dev_test_model_df['difficult'] == 0].reset_index(drop=True)

print('Split counts:', len(train_ids), len(val_ids), len(dev_test_ids), len(test_ids))
print('train-val overlap:', len(set(train_ids) & set(val_ids)))
print('train-dev_test overlap:', len(set(train_ids) & set(dev_test_ids)))
print('val-dev_test overlap:', len(set(val_ids) & set(dev_test_ids)))
print('train-test overlap:', len(set(train_ids) & set(test_ids)))
print('val-test overlap:', len(set(val_ids) & set(test_ids)))
print('dev_test-test overlap:', len(set(dev_test_ids) & set(test_ids)))

display(train_model_df_ignoring_difficult.head())

## 4. Exploratory Data Analysis

This section validates dataset quality before training: class balance, annotation flags, and bounding-box geometry.

These checks justify design choices for training mode, augmentation, and evaluation focus.

In [ ]:
trainval_images = pd.concat([train_images, val_images], ignore_index=True).drop_duplicates(subset=['image_id'])
trainval_objects = pd.concat([train_objects, val_objects], ignore_index=True)

if trainval_objects.empty:
    raise RuntimeError('No train/val annotations found. Check annotation paths and split files.')

eda_df = trainval_objects.copy()
obj_counts = eda_df.groupby('cls').size().reindex(VOC_CLASSES, fill_value=0).rename('object_count')
img_counts = eda_df.groupby('cls')['image_id'].nunique().reindex(VOC_CLASSES, fill_value=0).rename('image_count')
class_stats = pd.concat([img_counts, obj_counts], axis=1).sort_values('object_count', ascending=False)

class_stats

In [ ]:
plt.figure(figsize=(12, 5))
order = class_stats.index.tolist()
sns.barplot(data=class_stats.reset_index(), x='cls', y='object_count', order=order, color='#2a9d8f')
plt.xticks(rotation=45, ha='right')
plt.title('Object Count per Class (train + val)')
plt.tight_layout()
plt.show()

plt.figure(figsize=(12, 5))
sns.barplot(data=class_stats.reset_index(), x='cls', y='image_count', order=order, color='#e76f51')
plt.xticks(rotation=45, ha='right')
plt.title('Image Count per Class (train + val)')
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Plot 1: Per-class AP@0.5 comparison
per_class_df[['val_ap50', 'test_ap50']].plot(kind='bar', ax=axes[0], width=0.8)
axes[0].set_title(f'Per-class AP@0.5 ({best_experiment})')
axes[0].set_ylabel('AP@0.5')
axes[0].set_ylim(0, 1.05)
axes[0].grid(axis='y', linestyle='--', alpha=0.7)
axes[0].legend(['Validation', 'Test'])

# Plot 2: Overall mAP comparison
splits = ['val', 'dev_test', 'test']
maps = [overall_val_map, overall_dev_test_map, overall_test_map]
bars = axes[1].bar(splits, maps, color=['blue', 'orange', 'green'], alpha=0.7)
axes[1].set_title('Overall mAP@0.5 by Split')
axes[1].set_ylabel('mAP@0.5')
axes[1].set_ylim(0, 1.05)
axes[1].grid(axis='y', linestyle='--', alpha=0.7)

for bar in bars:
    yval = bar.get_height()
    axes[1].text(bar.get_x() + bar.get_width()/2, yval + 0.02, f'{yval:.4f}', ha='center', va='bottom')

plt.tight_layout()
plt.show()


### EDA: Augmentation Preview

This quick diagnostic visualizes random training images before and after augmentation, with bounding boxes overlaid in both views.

Use it to sanity-check whether spatial transforms (for example horizontal flip) keep boxes aligned with the objects.

In [ ]:
from matplotlib.patches import Rectangle
import torchvision.transforms.functional as TF


def draw_boxes(ax, boxes_df: pd.DataFrame, color: str = '#e63946', lw: float = 2.0) -> None:
    for row in boxes_df.itertuples(index=False):
        x = float(row.xmin)
        y = float(row.ymin)
        w = float(row.xmax - row.xmin + 1)
        h = float(row.ymax - row.ymin + 1)
        rect = Rectangle((x, y), w, h, fill=False, edgecolor=color, linewidth=lw)
        ax.add_patch(rect)



def random_augmentation_with_boxes(
    image: Image.Image,
    boxes_df: pd.DataFrame,
    flip_prob: float = 0.5,
    max_brightness_jitter: float = 0.25,
    max_contrast_jitter: float = 0.25,
):
    aug_image = image.copy()
    aug_boxes = boxes_df[['xmin', 'ymin', 'xmax', 'ymax']].copy().reset_index(drop=True)

    if random.random() < flip_prob:
        width = aug_image.width
        aug_image = TF.hflip(aug_image)
        old_xmin = aug_boxes['xmin'].copy()
        old_xmax = aug_boxes['xmax'].copy()
        aug_boxes['xmin'] = width - old_xmax - 1
        aug_boxes['xmax'] = width - old_xmin - 1

    brightness_factor = 1.0 + random.uniform(-max_brightness_jitter, max_brightness_jitter)
    contrast_factor = 1.0 + random.uniform(-max_contrast_jitter, max_contrast_jitter)
    aug_image = TF.adjust_brightness(aug_image, brightness_factor)
    aug_image = TF.adjust_contrast(aug_image, contrast_factor)

    return aug_image, aug_boxes



def show_augmented_samples(
    image_ids: list[str],
    objects_df: pd.DataFrame,
    image_dir: Path,
    n_samples: int = 4,
    random_state: int = 42,
):
    rng = random.Random(random_state)

    candidate_ids = [
        image_id for image_id in image_ids
        if image_id in set(objects_df['image_id']) and (image_dir / f'{image_id}.jpg').exists()
    ]

    if not candidate_ids:
        raise RuntimeError('No valid training images with boxes were found for augmentation preview.')

    n_samples = min(n_samples, len(candidate_ids))
    selected_ids = rng.sample(candidate_ids, n_samples)

    fig, axes = plt.subplots(n_samples, 2, figsize=(14, 4 * n_samples))
    if n_samples == 1:
        axes = np.array([axes])

    for row_idx, image_id in enumerate(selected_ids):
        image_path = image_dir / f'{image_id}.jpg'
        image = Image.open(image_path).convert('RGB')
        boxes_df = objects_df.loc[objects_df['image_id'] == image_id, ['xmin', 'ymin', 'xmax', 'ymax']]

        aug_image, aug_boxes = random_augmentation_with_boxes(image, boxes_df)

        ax_original = axes[row_idx, 0]
        ax_augmented = axes[row_idx, 1]

        ax_original.imshow(image)
        draw_boxes(ax_original, boxes_df)
        ax_original.set_title(f'Original | {image_id} | boxes={len(boxes_df)}')
        ax_original.axis('off')

        ax_augmented.imshow(aug_image)
        draw_boxes(ax_augmented, aug_boxes)
        ax_augmented.set_title(f'Augmented | {image_id} | boxes={len(aug_boxes)}')
        ax_augmented.axis('off')

    plt.tight_layout()
    plt.show()


show_augmented_samples(
    image_ids=train_ids,
    objects_df=train_model_df_ignoring_difficult,
    image_dir=TRAIN_IMAGE_DIR,
    n_samples=4,
    random_state=SEED,
)

In [ ]:
class VOCDetectionDataset(Dataset):
    def __init__(self, image_dir: Path, image_ids: list[str], objects_df: pd.DataFrame, transforms=None):
        self.image_dir = image_dir
        self.image_ids = list(image_ids)
        self.transforms = transforms
        self.objects_by_image = {}

        if not objects_df.empty:
            for image_id, group in objects_df.groupby('image_id'):
                self.objects_by_image[image_id] = group.copy()

    def __len__(self):
        return len(self.image_ids)

    def __getitem__(self, idx):
        image_id = self.image_ids[idx]
        image_path = self.image_dir / f'{image_id}.jpg'
        image = Image.open(image_path).convert('RGB')

        rows = self.objects_by_image.get(image_id)
        if rows is None or rows.empty:
            boxes = torch.zeros((0, 4), dtype=torch.float32)
            labels = torch.zeros((0,), dtype=torch.int64)
            area = torch.zeros((0,), dtype=torch.float32)
        else:
            boxes = torch.tensor(rows[['xmin', 'ymin', 'xmax', 'ymax']].to_numpy(), dtype=torch.float32)
            labels = torch.tensor(rows['class_id'].to_numpy(), dtype=torch.int64)
            area = torch.tensor(rows['box_area'].to_numpy(), dtype=torch.float32)

        target = {
            'boxes': boxes,
            'labels': labels,
            'area': area,
            'iscrowd': torch.zeros((boxes.shape[0],), dtype=torch.int64),
            'image_id': image_id,
        }

        if self.transforms is not None:
            image, target = self.transforms(image, target)

        return image, target



def collate_fn(batch):
    images, targets = zip(*batch)
    return list(images), list(targets)


train_ds = VOCDetectionDataset(TRAIN_IMAGE_DIR, train_ids, train_model_df_ignoring_difficult)
val_ds = VOCDetectionDataset(VAL_IMAGE_DIR, val_ids, val_model_df_ignoring_difficult)
dev_test_ds = VOCDetectionDataset(DEV_TEST_IMAGE_DIR, dev_test_ids, dev_test_model_df_ignoring_difficult)
test_ds = VOCDetectionDataset(TEST_IMAGE_DIR, test_ids, test_objects)

print('Dataset sizes:', len(train_ds), len(val_ds), len(dev_test_ds), len(test_ds))

In [ ]:
def compute_iou_xyxy(box_a: np.ndarray, box_b: np.ndarray) -> float:
    xa1, ya1, xa2, ya2 = box_a
    xb1, yb1, xb2, yb2 = box_b

    inter_x1 = max(xa1, xb1)
    inter_y1 = max(ya1, yb1)
    inter_x2 = min(xa2, xb2)
    inter_y2 = min(ya2, yb2)

    inter_w = max(0.0, inter_x2 - inter_x1 + 1)
    inter_h = max(0.0, inter_y2 - inter_y1 + 1)
    inter_area = inter_w * inter_h

    area_a = max(0.0, xa2 - xa1 + 1) * max(0.0, ya2 - ya1 + 1)
    area_b = max(0.0, xb2 - xb1 + 1) * max(0.0, yb2 - yb1 + 1)

    union = area_a + area_b - inter_area
    return 0.0 if union <= 0 else inter_area / union



def ap_from_pr(precisions: np.ndarray, recalls: np.ndarray) -> float:
    mrec = np.concatenate(([0.0], recalls, [1.0]))
    mpre = np.concatenate(([0.0], precisions, [0.0]))

    for i in range(len(mpre) - 1, 0, -1):
        mpre[i - 1] = max(mpre[i - 1], mpre[i])

    idx = np.where(mrec[1:] != mrec[:-1])[0]
    return float(np.sum((mrec[idx + 1] - mrec[idx]) * mpre[idx + 1]))



def evaluate_map50(pred_df: pd.DataFrame, gt_df: pd.DataFrame, classes: list[str], iou_thresh: float = 0.5) -> dict:
    if pred_df.empty or gt_df.empty:
        return {'map50': 0.0, 'per_class_ap': {cls: 0.0 for cls in classes}}

    per_class_ap = {}
    valid_classes = []

    for cls in classes:
        gt_cls = gt_df[gt_df['cls'] == cls]
        if gt_cls.empty:
            continue

        valid_classes.append(cls)
        pred_cls = pred_df[pred_df['cls'] == cls].sort_values('score', ascending=False)

        gt_by_image = {}
        for image_id, group in gt_cls.groupby('image_id'):
            gt_by_image[image_id] = {
                'boxes': group[['xmin', 'ymin', 'xmax', 'ymax']].to_numpy(dtype=float),
                'matched': np.zeros(len(group), dtype=bool),
            }

        tps = np.zeros(len(pred_cls), dtype=float)
        fps = np.zeros(len(pred_cls), dtype=float)

        for i, row in enumerate(pred_cls.itertuples(index=False)):
            image_state = gt_by_image.get(row.image_id)
            if image_state is None or len(image_state['boxes']) == 0:
                fps[i] = 1.0
                continue

            pred_box = np.array([row.xmin, row.ymin, row.xmax, row.ymax], dtype=float)
            ious = np.array([compute_iou_xyxy(pred_box, gt_box) for gt_box in image_state['boxes']])
            best_idx = int(np.argmax(ious))
            best_iou = ious[best_idx]

            if best_iou >= iou_thresh and not image_state['matched'][best_idx]:
                tps[i] = 1.0
                image_state['matched'][best_idx] = True
            else:
                fps[i] = 1.0

        cum_tp = np.cumsum(tps)
        cum_fp = np.cumsum(fps)
        recalls = cum_tp / max(1, len(gt_cls))
        precisions = cum_tp / np.maximum(cum_tp + cum_fp, 1e-12)
        per_class_ap[cls] = ap_from_pr(precisions, recalls) if len(pred_cls) > 0 else 0.0

    if not valid_classes:
        return {'map50': 0.0, 'per_class_ap': {cls: 0.0 for cls in classes}}

    map50 = float(np.mean([per_class_ap[cls] for cls in valid_classes]))
    return {'map50': map50, 'per_class_ap': per_class_ap}



def ensure_prediction_schema(df: pd.DataFrame, source_model: str, split_name: str) -> pd.DataFrame:
    out = df.copy()
    out['source_model'] = source_model
    out['split'] = split_name
    for col in PREDICTION_COLUMNS:
        if col not in out.columns:
            out[col] = np.nan
    out = out[PREDICTION_COLUMNS]
    out['class_id'] = out['class_id'].astype('Int64')
    return out

## 5. Faster R-CNN Setup and Training Utilities

This section defines the detector architecture, preprocessing presets, training modes, checkpointing, and mAP@0.5 evaluation utilities.

Training modes implemented:
- Head-only fine-tuning
- Full fine-tuning
- Incremental unfreezing

In [ ]:
import gc
import time

import torchvision
import torchvision.transforms as T
from torchvision.models.detection import (
    FasterRCNN_MobileNet_V3_Large_320_FPN_Weights,
    FasterRCNN_MobileNet_V3_Large_FPN_Weights,
    FasterRCNN_ResNet50_FPN_V2_Weights,
    FasterRCNN_ResNet50_FPN_Weights,
)
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor


VOC_IMAGE_MEAN = (0.485, 0.456, 0.406)
VOC_IMAGE_STD = (0.229, 0.224, 0.225)
DEFAULT_BASE_RCNN_MODEL = 'fasterrcnn_resnet50_fpn_v2'

BASE_RCNN_MODELS = {
    'fasterrcnn_resnet50_fpn': {
        'factory': torchvision.models.detection.fasterrcnn_resnet50_fpn,
        'weights': FasterRCNN_ResNet50_FPN_Weights.DEFAULT,
    },
    'fasterrcnn_resnet50_fpn_v2': {
        'factory': torchvision.models.detection.fasterrcnn_resnet50_fpn_v2,
        'weights': FasterRCNN_ResNet50_FPN_V2_Weights.DEFAULT,
    },
    'fasterrcnn_mobilenet_v3_large_fpn': {
        'factory': torchvision.models.detection.fasterrcnn_mobilenet_v3_large_fpn,
        'weights': FasterRCNN_MobileNet_V3_Large_FPN_Weights.DEFAULT,
    },
    'fasterrcnn_mobilenet_v3_large_320_fpn': {
        'factory': torchvision.models.detection.fasterrcnn_mobilenet_v3_large_320_fpn,
        'weights': FasterRCNN_MobileNet_V3_Large_320_FPN_Weights.DEFAULT,
    },
}


class AddGaussianNoise:
    def __init__(self, mean: float = 0.0, std: float = 0.03):
        self.mean = mean
        self.std = std

    def __call__(self, image: torch.Tensor) -> torch.Tensor:
        noise = torch.randn_like(image) * self.std + self.mean
        return torch.clamp(image + noise, 0.0, 1.0)


class DetectionTransform:
    def __init__(self, augment: bool = False, normalize: bool = False, noise: bool = False):
        self.augment = augment
        self.normalize = normalize
        self.noise = noise
        self.to_tensor = T.ToTensor()
        self.normalize_op = T.Normalize(VOC_IMAGE_MEAN, VOC_IMAGE_STD)
        self.color_jitter = T.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.1, hue=0.02)
        self.noise_op = AddGaussianNoise(std=0.03)

    def _horizontal_flip(self, image, target):
        width = image.width
        boxes = target['boxes'].clone()
        if boxes.numel() > 0:
            xmin = boxes[:, 0].clone()
            xmax = boxes[:, 2].clone()
            boxes[:, 0] = width - xmax - 1
            boxes[:, 2] = width - xmin - 1
            target = dict(target)
            target['boxes'] = boxes
        return image.transpose(method=Image.FLIP_LEFT_RIGHT), target

    def __call__(self, image, target):
        if self.augment:
            if random.random() < 0.5:
                image, target = self._horizontal_flip(image, target)
            image = self.color_jitter(image)

        image = self.to_tensor(image)

        if self.noise:
            image = self.noise_op(image)

        if self.normalize:
            image = self.normalize_op(image)

        return image, target


PREPROCESSING_PRESETS = {
    'baseline': {'augment': False, 'normalize': True, 'noise': False},
    'augmentation': {'augment': True, 'normalize': True, 'noise': False},
    'noise': {'augment': False, 'normalize': True, 'noise': True},
    'preprocessing': {'augment': True, 'normalize': True, 'noise': False},
    'preprocessing_noise': {'augment': True, 'normalize': True, 'noise': True},
}

EXPERIMENT_LIBRARY = {
    'head_only_baseline': {
        'base_model_name': DEFAULT_BASE_RCNN_MODEL,
        'preprocessing_mode': 'baseline',
        'training_mode': 'head_only',
        'epochs': 10,
        'batch_size': 2,
        'lr': 1e-4,
        'weight_decay': 1e-4,
    },
    'augmentation_head_only': {
        'base_model_name': DEFAULT_BASE_RCNN_MODEL,
        'preprocessing_mode': 'augmentation',
        'training_mode': 'head_only',
        'epochs': 10,
        'batch_size': 2,
        'lr': 1e-4,
        'weight_decay': 1e-4,
    },
    'preprocessing_noise_head_only': {
        'base_model_name': DEFAULT_BASE_RCNN_MODEL,
        'preprocessing_mode': 'preprocessing_noise',
        'training_mode': 'head_only',
        'epochs': 10,
        'batch_size': 2,
        'lr': 1e-4,
        'weight_decay': 1e-4,
    },
}

EXPERIMENTS_TO_RUN = [
    'head_only_baseline',
    'augmentation_head_only',
    'preprocessing_noise_head_only',
]

EXPERIMENT_RUNS_DIR = ARTIFACTS_DIR / 'experiment_runs'
EXPERIMENT_RUNS_DIR.mkdir(parents=True, exist_ok=True)


def sync_cuda():
    if torch.cuda.is_available():
        torch.cuda.synchronize()



def build_preprocessing_transform(preprocessing_mode: str, stage: str = 'train') -> DetectionTransform:
    preset = PREPROCESSING_PRESETS[preprocessing_mode]
    augment = bool(preset['augment']) if stage == 'train' else False
    normalize = bool(preset['normalize'])
    noise = bool(preset['noise']) if stage == 'train' else False
    return DetectionTransform(augment=augment, normalize=normalize, noise=noise)


DATASET_CACHE = {}



def build_datasets_and_loaders(preprocessing_mode: str, batch_size: int):
    cache_key = (preprocessing_mode, batch_size)
    if cache_key in DATASET_CACHE:
        return DATASET_CACHE[cache_key]

    train_transform = build_preprocessing_transform(preprocessing_mode, stage='train')
    eval_transform = build_preprocessing_transform(preprocessing_mode, stage='eval')

    train_ds = VOCDetectionDataset(TRAIN_IMAGE_DIR, train_ids, train_model_df_ignoring_difficult, transforms=train_transform)
    val_ds = VOCDetectionDataset(VAL_IMAGE_DIR, val_ids, val_model_df_ignoring_difficult, transforms=eval_transform)
    dev_test_ds = VOCDetectionDataset(DEV_TEST_IMAGE_DIR, dev_test_ids, dev_test_model_df_ignoring_difficult, transforms=eval_transform)
    test_ds = VOCDetectionDataset(TEST_IMAGE_DIR, test_ids, test_objects, transforms=eval_transform)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, collate_fn=collate_fn)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, collate_fn=collate_fn)
    dev_test_loader = DataLoader(dev_test_ds, batch_size=batch_size, shuffle=False, collate_fn=collate_fn)
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False, collate_fn=collate_fn)

    DATASET_CACHE[cache_key] = (train_ds, val_ds, dev_test_ds, test_ds, train_loader, val_loader, dev_test_loader, test_loader)
    return DATASET_CACHE[cache_key]



def build_faster_rcnn_model(num_classes: int, base_model_name: str = DEFAULT_BASE_RCNN_MODEL) -> torch.nn.Module:
    if base_model_name not in BASE_RCNN_MODELS:
        raise KeyError(f'Unknown base RCNN model: {base_model_name}')

    model_spec = BASE_RCNN_MODELS[base_model_name]
    model = model_spec['factory'](weights=model_spec['weights'])
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)
    return model



def freeze_all_parameters(model: torch.nn.Module) -> None:
    for parameter in model.parameters():
        parameter.requires_grad = False



def set_head_only_trainable(model: torch.nn.Module) -> None:
    freeze_all_parameters(model)
    for parameter in model.roi_heads.box_predictor.parameters():
        parameter.requires_grad = True



def set_full_finetune_trainable(model: torch.nn.Module) -> None:
    for parameter in model.parameters():
        parameter.requires_grad = True



def ordered_backbone_parameter_names(model: torch.nn.Module, total_layers: int = 50) -> list[str]:
    names = [name for name, _ in model.named_parameters() if name.startswith('backbone')]
    names = list(reversed(names))
    return names[:total_layers]



def set_incremental_trainable(
    model: torch.nn.Module,
    epoch_index: int,
    total_layers: int = 50,
    layers_per_epoch: int = 10,
) -> list[str]:
    freeze_all_parameters(model)
    for parameter in model.roi_heads.box_predictor.parameters():
        parameter.requires_grad = True

    schedule = ordered_backbone_parameter_names(model, total_layers=total_layers)
    layers_to_unfreeze = min(total_layers, (epoch_index + 1) * layers_per_epoch)
    active_names = set(schedule[:layers_to_unfreeze])

    for name, parameter in model.named_parameters():
        if name in active_names:
            parameter.requires_grad = True

    return schedule[:layers_to_unfreeze]



def count_trainable_parameters(model: torch.nn.Module) -> int:
    return sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)



def targets_to_device(targets: list[dict], device: torch.device) -> list[dict]:
    moved_targets = []
    for target in targets:
        moved_targets.append({
            'boxes': target['boxes'].to(device),
            'labels': (target['labels'] + 1).to(device),
            'area': target['area'].to(device),
            'iscrowd': target['iscrowd'].to(device),
        })
    return moved_targets



def filter_model_outputs(output: dict, score_threshold: float = 0.05, max_detections: int = 50) -> dict:
    scores = output['scores'].detach().cpu().numpy()
    keep = scores >= score_threshold

    if keep.sum() == 0:
        return {
            'boxes': np.empty((0, 4), dtype=float),
            'scores': np.empty((0,), dtype=float),
            'labels': np.empty((0,), dtype=int),
        }

    boxes = output['boxes'].detach().cpu().numpy()[keep][:max_detections]
    scores = scores[keep][:max_detections]
    labels = output['labels'].detach().cpu().numpy()[keep][:max_detections] - 1

    return {
        'boxes': boxes,
        'scores': scores,
        'labels': labels,
    }



def save_checkpoint(model, optimizer, epoch: int, path: Path, extra_state: dict | None = None) -> None:
    checkpoint = {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'seed': SEED,
    }
    if extra_state:
        checkpoint.update(extra_state)
    torch.save(checkpoint, path)



def load_checkpoint(path: Path, map_location=DEVICE):
    if not path.exists():
        return None
    return torch.load(path, map_location=map_location)



def train_one_epoch(model, loader, optimizer, device: torch.device) -> tuple[float, float, float]:
    model.train()
    running_loss = 0.0
    batch_times = []
    sync_cuda()
    start_time = time.perf_counter()

    for images, targets in loader:
        batch_start = time.perf_counter()
        images = [image.to(device) for image in images]
        targets = targets_to_device(targets, device)

        loss_dict = model(images, targets)
        loss = sum(loss_dict.values())
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        sync_cuda()
        batch_times.append(time.perf_counter() - batch_start)
        running_loss += float(loss.item())

    sync_cuda()
    elapsed = time.perf_counter() - start_time
    mean_batch_time = float(np.mean(batch_times)) if batch_times else 0.0
    return running_loss / max(1, len(loader)), elapsed, mean_batch_time



def predict_to_df(model, loader, split_name: str, score_threshold: float = 0.05, max_detections: int = 50) -> tuple[pd.DataFrame, float]:
    model.eval()
    rows = []
    sync_cuda()
    start_time = time.perf_counter()

    with torch.no_grad():
        for images, targets in loader:
            image_ids = [target['image_id'] for target in targets]
            images_gpu = [image.to(DEVICE) for image in images]
            outputs = model(images_gpu)
            sync_cuda()

            for image_id, output in zip(image_ids, outputs):
                filtered = filter_model_outputs(output, score_threshold=score_threshold, max_detections=max_detections)
                for box, score, class_id in zip(filtered['boxes'], filtered['scores'], filtered['labels']):
                    if class_id < 0 or class_id >= len(VOC_CLASSES):
                        continue
                    rows.append({
                        'image_id': image_id,
                        'class_id': int(class_id),
                        'cls': ID_TO_CLASS[int(class_id)],
                        'xmin': float(box[0]),
                        'ymin': float(box[1]),
                        'xmax': float(box[2]),
                        'ymax': float(box[3]),
                        'score': float(score),
                    })

    sync_cuda()
    elapsed = time.perf_counter() - start_time
    return ensure_prediction_schema(pd.DataFrame(rows), source_model='fasterrcnn', split_name=split_name), elapsed



def tune_score_threshold(
    model,
    loader,
    gt_df: pd.DataFrame,
    classes: list[str],
    candidate_thresholds: list[float] | None = None,
    max_detections: int = 50,
) -> tuple[float, float, pd.DataFrame, float]:
    if candidate_thresholds is None:
        candidate_thresholds = [round(x, 2) for x in np.arange(0.05, 0.55, 0.05)]

    tuning_rows = []
    best_threshold = candidate_thresholds[0]
    best_map50 = -1.0
    sync_cuda()
    start_time = time.perf_counter()

    for threshold in candidate_thresholds:
        val_pred, inference_seconds = predict_to_df(
            model,
            loader,
            split_name='val',
            score_threshold=threshold,
            max_detections=max_detections,
        )
        metrics = evaluate_map50(val_pred, gt_df, classes)
        map50 = float(metrics['map50'])

        tuning_rows.append({
            'score_threshold': threshold,
            'map50': map50,
            'val_inference_seconds': inference_seconds,
        })
        del val_pred, metrics
        if map50 > best_map50:
            best_map50 = map50
            best_threshold = threshold

    sync_cuda()
    elapsed = time.perf_counter() - start_time
    tuning_df = pd.DataFrame(tuning_rows).sort_values('score_threshold').reset_index(drop=True)
    return best_threshold, best_map50, tuning_df, elapsed



def train_and_evaluate_experiment(experiment_name: str, config: dict) -> tuple[dict, pd.DataFrame]:
    train_ds, val_ds, dev_test_ds, test_ds, train_loader, val_loader, dev_test_loader, test_loader = build_datasets_and_loaders(
        config['preprocessing_mode'],
        config['batch_size'],
    )

    model = build_faster_rcnn_model(len(VOC_CLASSES) + 1, base_model_name=config.get('base_model_name', DEFAULT_BASE_RCNN_MODEL))
    model.to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=config['lr'], weight_decay=config['weight_decay'])
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=config['epochs'])

    if config['training_mode'] == 'head_only':
        set_head_only_trainable(model)
    elif config['training_mode'] == 'full_finetune':
        set_full_finetune_trainable(model)
    elif config['training_mode'] == 'incremental':
        set_head_only_trainable(model)
    else:
        raise ValueError(f"Unknown training_mode: {config['training_mode']}")

    trainable_params = count_trainable_parameters(model)
    total_params = sum(parameter.numel() for parameter in model.parameters())
    print(f"\n=== Experiment: {experiment_name} ===")
    print(f"Base model: {config.get('base_model_name', DEFAULT_BASE_RCNN_MODEL)}")
    print(f"Preprocessing: {config['preprocessing_mode']} | Training: {config['training_mode']}")
    print(f"Trainable parameters: {trainable_params:,} / {total_params:,}")

    checkpoint_dir = EXPERIMENT_RUNS_DIR / experiment_name
    checkpoint_dir.mkdir(parents=True, exist_ok=True)
    checkpoint_path = checkpoint_dir / 'best_checkpoint.pt'

    epoch_rows = []
    best_map50 = -1.0
    best_threshold = 0.05
    patience = 3
    epochs_without_improvement = 0
    total_train_start = time.perf_counter()

    for epoch in range(config['epochs']):
        if config['training_mode'] == 'incremental':
            active_layers = set_incremental_trainable(
                model,
                epoch_index=epoch,
                total_layers=config.get('incremental_layers_total', 50),
                layers_per_epoch=config.get('incremental_layers_per_epoch', 10),
            )
            print(f"Epoch {epoch + 1}: incremental backbone layers active = {len(active_layers)}")

        epoch_loss, epoch_seconds, batch_seconds = train_one_epoch(model, train_loader, optimizer, DEVICE)
        scheduler.step()
        val_pred, val_seconds = predict_to_df(model, val_loader, split_name='val', score_threshold=0.05)
        dev_test_pred, dev_test_seconds = predict_to_df(model, dev_test_loader, split_name='dev_test', score_threshold=0.05)
        val_metrics = evaluate_map50(val_pred, val_model_df_ignoring_difficult, VOC_CLASSES)
        dev_test_metrics = evaluate_map50(dev_test_pred, dev_test_model_df_ignoring_difficult, VOC_CLASSES)
        val_map50 = float(val_metrics['map50'])
        dev_test_map50 = float(dev_test_metrics['map50'])
        del val_pred, dev_test_pred, val_metrics, dev_test_metrics

        epoch_rows.append({
            'experiment': experiment_name,
            'epoch': epoch + 1,
            'loss': epoch_loss,
            'epoch_seconds': epoch_seconds,
            'batch_seconds': batch_seconds,
            'val_map50': val_map50,
            'dev_test_map50': dev_test_map50,
            'val_inference_seconds': val_seconds,
            'dev_test_inference_seconds': dev_test_seconds,
        })

        print(
            f"Epoch {epoch + 1}/{config['epochs']} | loss={epoch_loss:.4f} | "
            f"train_time={epoch_seconds:.1f}s | val_time={val_seconds:.1f}s | "
            f"dev_test_time={dev_test_seconds:.1f}s | val_mAP@0.5={val_map50:.4f} | dev_test_mAP@0.5={dev_test_map50:.4f}"
        )

        if val_map50 >= best_map50:
            best_map50 = val_map50
            epochs_without_improvement = 0
            save_checkpoint(model, optimizer, epoch, checkpoint_path, extra_state={'experiment': experiment_name})
            print(f"Saved best checkpoint to {checkpoint_path}")
        else:
            epochs_without_improvement += 1

        if epochs_without_improvement >= patience:
            print(f"Early stopping triggered after {epoch + 1} epochs")
            break

    total_train_seconds = time.perf_counter() - total_train_start

    checkpoint = load_checkpoint(checkpoint_path)
    if checkpoint is not None:
        model.load_state_dict(checkpoint['model_state_dict'])
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        del checkpoint

    best_threshold, tuned_val_map50, tuning_df, tuning_seconds = tune_score_threshold(
        model,
        val_loader,
        val_model_df_ignoring_difficult,
        VOC_CLASSES,
    )

    val_pred_df, val_inference_seconds = predict_to_df(
        model,
        val_loader,
        split_name='val',
        score_threshold=best_threshold,
    )
    dev_test_pred_df, dev_test_inference_seconds = predict_to_df(
        model,
        dev_test_loader,
        split_name='dev_test',
        score_threshold=best_threshold,
    )
    test_pred_df, test_inference_seconds = predict_to_df(
        model,
        test_loader,
        split_name='test',
        score_threshold=best_threshold,
    )

    final_val_metrics = evaluate_map50(val_pred_df, val_model_df_ignoring_difficult, VOC_CLASSES)
    final_dev_test_metrics = evaluate_map50(dev_test_pred_df, dev_test_model_df_ignoring_difficult, VOC_CLASSES)
    final_test_metrics = evaluate_map50(test_pred_df, test_objects, VOC_CLASSES)
    val_map50 = float(final_val_metrics['map50'])
    dev_test_map50 = float(final_dev_test_metrics['map50'])
    test_map50 = float(final_test_metrics['map50'])

    val_path = checkpoint_dir / f'{experiment_name}_val_predictions.csv'
    dev_test_path = checkpoint_dir / f'{experiment_name}_dev_test_predictions.csv'
    test_path = checkpoint_dir / f'{experiment_name}_test_predictions.csv'
    tuning_path = checkpoint_dir / f'{experiment_name}_threshold_tuning.csv'
    history_path = checkpoint_dir / f'{experiment_name}_history.csv'

    val_pred_df.to_csv(val_path, index=False)
    dev_test_pred_df.to_csv(dev_test_path, index=False)
    test_pred_df.to_csv(test_path, index=False)
    tuning_df.to_csv(tuning_path, index=False)
    history_df = pd.DataFrame(epoch_rows)
    history_df.to_csv(history_path, index=False)

    summary = {
        'experiment': experiment_name,
        'base_model_name': config.get('base_model_name', DEFAULT_BASE_RCNN_MODEL),
        'preprocessing_mode': config['preprocessing_mode'],
        'training_mode': config['training_mode'],
        'epochs': config['epochs'],
        'batch_size': config['batch_size'],
        'trainable_params': trainable_params,
        'total_params': total_params,
        'best_epoch_val_map50': best_map50,
        'tuned_val_map50': val_map50,
        'tuned_dev_test_map50': dev_test_map50,
        'tuned_test_map50': test_map50,
        'best_threshold': best_threshold,
        'train_seconds': total_train_seconds,
        'tuning_seconds': tuning_seconds,
        'val_inference_seconds': val_inference_seconds,
        'dev_test_inference_seconds': dev_test_inference_seconds,
        'test_inference_seconds': test_inference_seconds,
        'val_seconds_per_image': val_inference_seconds / max(1, len(val_ds)),
        'dev_test_seconds_per_image': dev_test_inference_seconds / max(1, len(dev_test_ds)),
        'test_seconds_per_image': test_inference_seconds / max(1, len(test_ds)),
        'history_path': str(history_path),
        'val_predictions_path': str(val_path),
        'dev_test_predictions_path': str(dev_test_path),
        'test_predictions_path': str(test_path),
        'tuning_path': str(tuning_path),
        'checkpoint_path': str(checkpoint_path),
    }

    print(
        f"Completed {experiment_name} | tuned_val_mAP@0.5={val_map50:.4f} | dev_test_mAP@0.5={dev_test_map50:.4f} | "
        f"test_mAP@0.5={test_map50:.4f} | best_threshold={best_threshold:.2f} | train_time={total_train_seconds:.1f}s | "
        f"val_inference={val_inference_seconds:.1f}s | dev_test_inference={dev_test_inference_seconds:.1f}s | test_inference={test_inference_seconds:.1f}s"
    )
    print(f"Saved predictions to {val_path}, {dev_test_path}, and {test_path}")

    del val_pred_df, dev_test_pred_df, test_pred_df, tuning_df, epoch_rows
    del model, optimizer, train_ds, val_ds, dev_test_ds, test_ds, train_loader, val_loader, dev_test_loader, test_loader
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return summary, history_df


print('Experiment runner is ready. Set EXPERIMENTS_TO_RUN in the next cell to launch the comparisons.')

### 5.1 Class-Imbalance Handling

This optional block enables class-balanced classification loss for full and incremental fine-tuning modes.

It helps stabilise training when frequent classes dominate gradient updates.

In [ ]:
import contextlib
import torch.nn.functional as F
import torchvision.models.detection.roi_heads as roi_heads_module


def build_balanced_class_weights(
    train_objects_df: pd.DataFrame,
    num_foreground_classes: int,
    background_weight: float = 1.0,
) -> torch.Tensor:
    counts = np.ones(num_foreground_classes, dtype=np.float64)
    observed_counts = train_objects_df['class_id'].value_counts().to_dict()

    for class_id in range(num_foreground_classes):
        counts[class_id] = max(1.0, float(observed_counts.get(class_id, 0)))

    total = float(np.sum(counts))
    balanced_fg = total / (num_foreground_classes * counts)

    # Keep the average foreground weight at 1.0 so optimizer settings remain stable.
    balanced_fg = balanced_fg / np.mean(balanced_fg)

    weights = np.concatenate(([background_weight], balanced_fg)).astype(np.float32)
    return torch.tensor(weights, dtype=torch.float32)


CLASS_BALANCED_WEIGHTS = build_balanced_class_weights(
    train_model_df_ignoring_difficult,
    num_foreground_classes=len(VOC_CLASSES),
    background_weight=1.0,
)

print('Class-balanced weights (index 0 is background):')
print(CLASS_BALANCED_WEIGHTS)


_original_fastrcnn_loss = roi_heads_module.fastrcnn_loss


def weighted_fastrcnn_loss(class_logits, box_regression, labels, regression_targets):
    _, box_loss = _original_fastrcnn_loss(
        class_logits,
        box_regression,
        labels,
        regression_targets,
    )
    labels_concat = torch.cat(labels, dim=0)
    if labels_concat.numel() == 0:
        cls_loss = class_logits.sum() * 0.0
    else:
        cls_loss = F.cross_entropy(
            class_logits,
            labels_concat,
            weight=CLASS_BALANCED_WEIGHTS.to(class_logits.device),
        )
    return cls_loss, box_loss


@contextlib.contextmanager
def maybe_enable_weighted_loss(enabled: bool):
    if not enabled:
        yield
        return

    roi_heads_module.fastrcnn_loss = weighted_fastrcnn_loss
    try:
        yield
    finally:
        roi_heads_module.fastrcnn_loss = _original_fastrcnn_loss


if '_base_train_and_evaluate_experiment' not in globals():
    _base_train_and_evaluate_experiment = train_and_evaluate_experiment


def train_and_evaluate_experiment(experiment_name: str, config: dict) -> tuple[dict, pd.DataFrame]:
    use_balanced_loss = config.get(
        'use_class_balanced_loss',
        config.get('training_mode') in {'full_finetune', 'incremental'},
    )
    with maybe_enable_weighted_loss(use_balanced_loss):
        return _base_train_and_evaluate_experiment(experiment_name, config)


for exp_name, exp_cfg in EXPERIMENT_LIBRARY.items():
    if exp_cfg.get('training_mode') in {'full_finetune', 'incremental'}:
        exp_cfg['use_class_balanced_loss'] = True
    else:
        exp_cfg.setdefault('use_class_balanced_loss', False)

print('Balanced class weighting enabled for full/incremental fine-tuning experiments.')

## 6. Experiment Suite (3 Explicit Runs)

Each experiment is now executed in its own code cell (no for-loop), which improves reproducibility, debugging, and report presentation.

Run the setup cell first, then execute the five experiments below.

In [ ]:
EXPERIMENT_RESULTS = []
EXPERIMENT_HISTORY = {}

def run_single_experiment(experiment_name: str):
    if experiment_name not in EXPERIMENT_LIBRARY:
        raise KeyError(f'Unknown experiment: {experiment_name}')

    summary, history_df = train_and_evaluate_experiment(
        experiment_name,
        EXPERIMENT_LIBRARY[experiment_name],
    )
    EXPERIMENT_RESULTS.append(summary)
    EXPERIMENT_HISTORY[experiment_name] = history_df
    return summary, history_df

print('Experiment containers are ready. Run the five experiment cells below in order.')

### 6.1 Experiment 1 - Head-Only Baseline

Objective: establish a lightweight baseline by training only the detection head while keeping the backbone frozen.

In [ ]:
summary_head_only, history_head_only = run_single_experiment('head_only_baseline')
display(pd.DataFrame([summary_head_only]))
display(history_head_only.tail())

### 6.2 Experiment 2 - Augmentation Head-Only


In [ ]:
summary_preproc_ft, history_preproc_ft = run_single_experiment('augmentation_head_only')
display(pd.DataFrame([summary_preproc_ft]))
display(history_preproc_ft.tail())

### 6.3 Experiment 3 - Preprocessing + Noise Head-Only


In [ ]:
summary_preproc_noise_ft, history_preproc_noise_ft = run_single_experiment('preprocessing_noise_head_only')
display(pd.DataFrame([summary_preproc_noise_ft]))
display(history_preproc_noise_ft.tail())

### 6.4 Aggregate Experiment Results


In [ ]:
results_df = pd.DataFrame(EXPERIMENT_RESULTS).sort_values('tuned_val_map50', ascending=False).reset_index(drop=True)
results_path = EXPERIMENT_RUNS_DIR / 'experiment_summary.csv'
results_df.to_csv(results_path, index=False)

display(
    results_df[
        [
            'experiment',
            'base_model_name',
            'preprocessing_mode',
            'training_mode',
            'epochs',
            'trainable_params',
            'best_epoch_val_map50',
            'tuned_val_map50',
            'tuned_dev_test_map50',
            'tuned_test_map50',
            'best_threshold',
            'train_seconds',
            'val_inference_seconds',
            'dev_test_inference_seconds',
            'test_inference_seconds',
        ]
    ]
)
print('Saved experiment summary to:', results_path)

### 6.5 Per-Experiment Visualisation


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Plot 1: Per-class AP@0.5 comparison
per_class_df[['val_ap50', 'test_ap50']].plot(kind='bar', ax=axes[0], width=0.8)
axes[0].set_title(f'Per-class AP@0.5 ({best_experiment})')
axes[0].set_ylabel('AP@0.5')
axes[0].set_ylim(0, 1.05)
axes[0].grid(axis='y', linestyle='--', alpha=0.7)
axes[0].legend(['Validation', 'Test'])

# Plot 2: Overall mAP comparison
splits = ['val', 'dev_test', 'test']
maps = [overall_val_map, overall_dev_test_map, overall_test_map]
bars = axes[1].bar(splits, maps, color=['blue', 'orange', 'green'], alpha=0.7)
axes[1].set_title('Overall mAP@0.5 by Split')
axes[1].set_ylabel('mAP@0.5')
axes[1].set_ylim(0, 1.05)
axes[1].grid(axis='y', linestyle='--', alpha=0.7)

for bar in bars:
    yval = bar.get_height()
    axes[1].text(bar.get_x() + bar.get_width()/2, yval + 0.02, f'{yval:.4f}', ha='center', va='bottom')

plt.tight_layout()
plt.show()


## 7. Evaluation and Visualisation (All Classes)

This section evaluates the best performing model across all 20 PASCAL VOC classes. We calculate AP@0.5 for each class individually and report the overall mean AP (mAP@0.5) across the `val`, `dev_test`, and `test` splits.

In [ ]:
if 'results_df' not in globals() or results_df.empty:
    raise RuntimeError('Run Section 6 first so experiment summaries are available.')

best_row = results_df.iloc[0]
best_experiment = best_row['experiment']
best_threshold = float(best_row['best_threshold'])

val_pred_path = Path(best_row['val_predictions_path'])
dev_test_pred_path = Path(best_row['dev_test_predictions_path']) if 'dev_test_predictions_path' in best_row and pd.notna(best_row['dev_test_predictions_path']) else None
test_pred_path = Path(best_row['test_predictions_path'])

val_pred_df = pd.read_csv(val_pred_path) if val_pred_path.exists() else pd.DataFrame(columns=PREDICTION_COLUMNS)
dev_test_pred_df = pd.read_csv(dev_test_pred_path) if dev_test_pred_path and dev_test_pred_path.exists() else pd.DataFrame(columns=PREDICTION_COLUMNS)
test_pred_df = pd.read_csv(test_pred_path) if test_pred_path.exists() else pd.DataFrame(columns=PREDICTION_COLUMNS)

def single_class_pr_ap(pred_df: pd.DataFrame, gt_df: pd.DataFrame, cls_name: str, iou_thresh: float = 0.5):
    pred_cls = pred_df[pred_df['cls'] == cls_name].sort_values('score', ascending=False).reset_index(drop=True)
    gt_cls = gt_df[gt_df['cls'] == cls_name].copy()

    if len(gt_cls) == 0:
        return {'ap50': 0.0, 'n_gt': 0, 'n_pred': len(pred_cls), 'precisions': [], 'recalls': []}

    gt_by_image = {}
    for image_id, group in gt_cls.groupby('image_id'):
        gt_by_image[image_id] = {
            'boxes': group[['xmin', 'ymin', 'xmax', 'ymax']].to_numpy(dtype=float),
            'matched': np.zeros(len(group), dtype=bool),
        }

    tps = np.zeros(len(pred_cls), dtype=float)
    fps = np.zeros(len(pred_cls), dtype=float)

    for i, row in enumerate(pred_cls.itertuples(index=False)):
        state = gt_by_image.get(row.image_id)
        if state is None or len(state['boxes']) == 0:
            fps[i] = 1.0
            continue

        pred_box = np.array([row.xmin, row.ymin, row.xmax, row.ymax], dtype=float)
        ious = np.array([compute_iou_xyxy(pred_box, gt_box) for gt_box in state['boxes']])
        best_idx = int(np.argmax(ious))
        best_iou = ious[best_idx]

        if best_iou >= iou_thresh and not state['matched'][best_idx]:
            tps[i] = 1.0
            state['matched'][best_idx] = True
        else:
            fps[i] = 1.0

    cum_tp = np.cumsum(tps)
    cum_fp = np.cumsum(fps)
    n_gt = max(1, len(gt_cls))
    recalls = cum_tp / n_gt
    precisions = cum_tp / np.maximum(cum_tp + cum_fp, 1e-12)
    ap50 = ap_from_pr(precisions, recalls) if len(pred_cls) else 0.0

    return {
        'pred_sorted': pred_cls,
        'n_gt': len(gt_cls),
        'n_pred': len(pred_cls),
        'precisions': precisions,
        'recalls': recalls,
        'ap50': float(ap50),
    }

per_class_results = []

for cls_name in VOC_CLASSES:
    val_metrics = single_class_pr_ap(val_pred_df, val_model_df_ignoring_difficult, cls_name, iou_thresh=0.5)
    dev_test_metrics = single_class_pr_ap(dev_test_pred_df, dev_test_model_df_ignoring_difficult, cls_name, iou_thresh=0.5)
    test_metrics = single_class_pr_ap(test_pred_df, test_objects, cls_name, iou_thresh=0.5)
    
    per_class_results.append({
        'class': cls_name,
        'val_ap50': val_metrics['ap50'],
        'dev_test_ap50': dev_test_metrics['ap50'],
        'test_ap50': test_metrics['ap50'],
        'val_gt': val_metrics['n_gt'],
        'test_gt': test_metrics['n_gt']
    })

per_class_df = pd.DataFrame(per_class_results)
per_class_df.set_index('class', inplace=True)

overall_val_map = per_class_df['val_ap50'].mean()
overall_dev_test_map = per_class_df['dev_test_ap50'].mean()
overall_test_map = per_class_df['test_ap50'].mean()

print(f"=== Multi-class Evaluation ({best_experiment}) ===")
print(f"Operating threshold: {best_threshold:.2f}")
print(f"Overall mAP@0.5 (val):      {overall_val_map:.4f}")
print(f"Overall mAP@0.5 (dev_test): {overall_dev_test_map:.4f}")
print(f"Overall mAP@0.5 (test):     {overall_test_map:.4f}")
print("\nPer-class AP@0.5:")
display(per_class_df.round(4))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Plot 1: Per-class AP@0.5 comparison
per_class_df[['val_ap50', 'test_ap50']].plot(kind='bar', ax=axes[0], width=0.8)
axes[0].set_title(f'Per-class AP@0.5 ({best_experiment})')
axes[0].set_ylabel('AP@0.5')
axes[0].set_ylim(0, 1.05)
axes[0].grid(axis='y', linestyle='--', alpha=0.7)
axes[0].legend(['Validation', 'Test'])

# Plot 2: Overall mAP comparison
splits = ['val', 'dev_test', 'test']
maps = [overall_val_map, overall_dev_test_map, overall_test_map]
bars = axes[1].bar(splits, maps, color=['blue', 'orange', 'green'], alpha=0.7)
axes[1].set_title('Overall mAP@0.5 by Split')
axes[1].set_ylabel('mAP@0.5')
axes[1].set_ylim(0, 1.05)
axes[1].grid(axis='y', linestyle='--', alpha=0.7)

for bar in bars:
    yval = bar.get_height()
    axes[1].text(bar.get_x() + bar.get_width()/2, yval + 0.02, f'{yval:.4f}', ha='center', va='bottom')

plt.tight_layout()
plt.show()


## 8. Predict and Export

Once training completes, this section reports the best experiment and exposes the prediction CSV paths for downstream ensemble/server evaluation workflows.

In [ ]:
if 'EXPERIMENT_RESULTS' not in globals() or not EXPERIMENT_RESULTS:
    raise RuntimeError('Run the experiment suite first so the notebook has results to export.')

results_df = pd.DataFrame(EXPERIMENT_RESULTS).sort_values('tuned_val_map50', ascending=False).reset_index(drop=True)
best_row = results_df.iloc[0]
best_experiment = best_row['experiment']

print('Best experiment:', best_experiment)
print('Preprocessing mode:', best_row['preprocessing_mode'])
print('Training mode:', best_row['training_mode'])
print('Validation mAP@0.5:', round(float(best_row['tuned_val_map50']), 4))
print('Best threshold:', f"{float(best_row['best_threshold']):.2f}")
print('Train time (seconds):', round(float(best_row['train_seconds']), 1))
print('Validation inference time (seconds):', round(float(best_row['val_inference_seconds']), 1))
print('Test inference time (seconds):', round(float(best_row['test_inference_seconds']), 1))
print('Validation predictions:', best_row['val_predictions_path'])
print('Test predictions:', best_row['test_predictions_path'])
print('History CSV:', best_row['history_path'])
print('Checkpoint:', best_row['checkpoint_path'])

display(results_df)
display(EXPERIMENT_HISTORY[best_experiment].tail())

## 9. Base Model Comparison (No Fine-Tuning)

This section evaluates a few pretrained TorchVision Faster R-CNN variants on the same VOC validation/test splits.

Use the validation split to choose the best architecture or fine-tuning recipe. That is not data leakage. It becomes leakage only if the test split influences model selection or hyperparameter tuning.

A safe workflow is:
- compare candidate RCNN variants on train/val only
- pick the best candidate using validation metrics
- fine-tune that candidate
- evaluate once on the untouched test split, or submit to the official VOC server

This section also compares the best fine-tuned experiment against a zero-shot COCO baseline so you can see how much gain comes from VOC adaptation versus the pretrained detector alone.

In [ ]:
from pathlib import Path
import time

import torchvision
from torchvision.models.detection import FasterRCNN_ResNet50_FPN_Weights

if 'results_df' not in globals() or results_df.empty:
    raise RuntimeError('Run Section 6 and Section 8 first so the best fine-tuned experiment is available.')

best_row = results_df.iloc[0]
best_experiment = str(best_row['experiment'])

best_val_path = Path(best_row['val_predictions_path'])
best_dev_test_path = Path(best_row['dev_test_predictions_path']) if 'dev_test_predictions_path' in best_row and pd.notna(best_row['dev_test_predictions_path']) else None
best_test_path = Path(best_row['test_predictions_path'])
if not best_val_path.exists() or not best_test_path.exists():
    raise FileNotFoundError('Best experiment prediction files were not found. Re-run Section 8 to regenerate them.')

best_val_pred = pd.read_csv(best_val_path)
best_dev_test_pred = pd.read_csv(best_dev_test_path) if best_dev_test_path and best_dev_test_path.exists() else pd.DataFrame(columns=PREDICTION_COLUMNS)
best_test_pred = pd.read_csv(best_test_path)

# Build evaluation loaders with baseline preprocessing for a fair no-finetuning comparison.
_, val_ds_eval, dev_test_ds_eval, test_ds_eval, _, val_loader_eval, dev_test_loader_eval, test_loader_eval = build_datasets_and_loaders(
    preprocessing_mode='baseline',
    batch_size=2,
)

weights = FasterRCNN_ResNet50_FPN_Weights.DEFAULT
base_model = torchvision.models.detection.fasterrcnn_resnet50_fpn(weights=weights)
base_model.to(DEVICE)
base_model.eval()

coco_categories = list(weights.meta['categories'])
coco_name_to_voc = {
    'airplane': 'aeroplane',
    'bicycle': 'bicycle',
    'bird': 'bird',
    'boat': 'boat',
    'bottle': 'bottle',
    'bus': 'bus',
    'car': 'car',
    'cat': 'cat',
    'chair': 'chair',
    'cow': 'cow',
    'dining table': 'diningtable',
    'dog': 'dog',
    'horse': 'horse',
    'motorcycle': 'motorbike',
    'person': 'person',
    'potted plant': 'pottedplant',
    'sheep': 'sheep',
    'couch': 'sofa',
    'train': 'train',
    'tv': 'tvmonitor',
}

coco_label_to_voc = {}
for coco_label, coco_name in enumerate(coco_categories):
    if coco_label == 0:
        continue
    voc_name = coco_name_to_voc.get(coco_name)
    if voc_name is not None:
        coco_label_to_voc[coco_label] = voc_name


def predict_base_model_to_voc_df(model, loader, split_name: str, score_threshold: float = 0.05, max_detections: int = 50):
    rows = []
    sync_cuda()
    start_time = time.perf_counter()

    with torch.no_grad():
        for images, targets in loader:
            image_ids = [target['image_id'] for target in targets]
            images_gpu = [image.to(DEVICE) for image in images]
            outputs = model(images_gpu)
            sync_cuda()

            for image_id, output in zip(image_ids, outputs):
                scores = output['scores'].detach().cpu().numpy()
                boxes = output['boxes'].detach().cpu().numpy()
                labels = output['labels'].detach().cpu().numpy()

                keep = scores >= score_threshold
                boxes = boxes[keep][:max_detections]
                scores_kept = scores[keep][:max_detections]
                labels_kept = labels[keep][:max_detections]

                for box, score, coco_label in zip(boxes, scores_kept, labels_kept):
                    voc_name = coco_label_to_voc.get(int(coco_label))
                    if voc_name is None:
                        continue
                    rows.append({
                        'image_id': image_id,
                        'class_id': int(CLASS_TO_ID[voc_name]),
                        'cls': voc_name,
                        'xmin': float(box[0]),
                        'ymin': float(box[1]),
                        'xmax': float(box[2]),
                        'ymax': float(box[3]),
                        'score': float(score),
                    })

    sync_cuda()
    elapsed = time.perf_counter() - start_time
    pred_df = ensure_prediction_schema(pd.DataFrame(rows), source_model='fasterrcnn_coco_base', split_name=split_name)
    return pred_df, elapsed


candidate_thresholds = [round(x, 2) for x in np.arange(0.05, 0.55, 0.05)]
threshold_rows = []
best_base_threshold = candidate_thresholds[0]
best_base_map50 = -1.0

for threshold in candidate_thresholds:
    base_val_pred_tmp, val_seconds_tmp = predict_base_model_to_voc_df(
        base_model,
        val_loader_eval,
        split_name='val',
        score_threshold=threshold,
        max_detections=50,
    )
    tmp_metrics = evaluate_map50(base_val_pred_tmp, val_model_df_ignoring_difficult, VOC_CLASSES)
    tmp_map50 = float(tmp_metrics['map50'])

    threshold_rows.append({
        'score_threshold': threshold,
        'val_map50': tmp_map50,
        'val_inference_seconds': val_seconds_tmp,
    })

    if tmp_map50 > best_base_map50:
        best_base_map50 = tmp_map50
        best_base_threshold = threshold

base_threshold_df = pd.DataFrame(threshold_rows)

base_val_pred, base_val_seconds = predict_base_model_to_voc_df(
    base_model,
    val_loader_eval,
    split_name='val',
    score_threshold=best_base_threshold,
    max_detections=50,
)
base_dev_test_pred, base_dev_test_seconds = predict_base_model_to_voc_df(
    base_model,
    dev_test_loader_eval,
    split_name='dev_test',
    score_threshold=best_base_threshold,
    max_detections=50,
)
base_test_pred, base_test_seconds = predict_base_model_to_voc_df(
    base_model,
    test_loader_eval,
    split_name='test',
    score_threshold=best_base_threshold,
    max_detections=50,
)

base_val_metrics = evaluate_map50(base_val_pred, val_model_df_ignoring_difficult, VOC_CLASSES)
base_dev_test_metrics = evaluate_map50(base_dev_test_pred, dev_test_model_df_ignoring_difficult, VOC_CLASSES)
base_test_metrics = evaluate_map50(base_test_pred, test_objects, VOC_CLASSES)

ft_val_metrics = evaluate_map50(best_val_pred, val_model_df_ignoring_difficult, VOC_CLASSES)
ft_dev_test_metrics = evaluate_map50(best_dev_test_pred, dev_test_model_df_ignoring_difficult, VOC_CLASSES)
ft_test_metrics = evaluate_map50(best_test_pred, test_objects, VOC_CLASSES)

base_val_person = evaluate_map50(base_val_pred, val_model_df_ignoring_difficult, ['person'])['map50']
base_dev_test_person = evaluate_map50(base_dev_test_pred, dev_test_model_df_ignoring_difficult, ['person'])['map50']
base_test_person = evaluate_map50(base_test_pred, test_objects, ['person'])['map50']
ft_val_person = evaluate_map50(best_val_pred, val_model_df_ignoring_difficult, ['person'])['map50']
ft_dev_test_person = evaluate_map50(best_dev_test_pred, dev_test_model_df_ignoring_difficult, ['person'])['map50']
ft_test_person = evaluate_map50(best_test_pred, test_objects, ['person'])['map50']

base_dir = EXPERIMENT_RUNS_DIR / 'base_model_zero_shot'
base_dir.mkdir(parents=True, exist_ok=True)

base_val_path = base_dir / 'base_model_val_predictions.csv'
base_dev_test_path = base_dir / 'base_model_dev_test_predictions.csv'
base_test_path = base_dir / 'base_model_test_predictions.csv'
base_tuning_path = base_dir / 'base_model_threshold_tuning.csv'
comparison_path = base_dir / 'base_vs_finetuned_comparison.csv'

base_val_pred.to_csv(base_val_path, index=False)
base_dev_test_pred.to_csv(base_dev_test_path, index=False)
base_test_pred.to_csv(base_test_path, index=False)
base_threshold_df.to_csv(base_tuning_path, index=False)

comparison_df = pd.DataFrame([
    {
        'model': f'best_finetuned::{best_experiment}',
        'threshold': float(best_row['best_threshold']),
        'val_map50_all_classes': float(ft_val_metrics['map50']),
        'dev_test_map50_all_classes': float(ft_dev_test_metrics['map50']),
        'test_map50_all_classes': float(ft_test_metrics['map50']),
        'val_ap50_person': float(ft_val_person),
        'dev_test_ap50_person': float(ft_dev_test_person),
        'test_ap50_person': float(ft_test_person),
        'val_seconds_per_image': float(best_row['val_seconds_per_image']),
        'dev_test_seconds_per_image': float(best_row['dev_test_seconds_per_image']),
        'test_seconds_per_image': float(best_row['test_seconds_per_image']),
    },
    {
        'model': 'base_coco_no_finetune',
        'threshold': float(best_base_threshold),
        'val_map50_all_classes': float(base_val_metrics['map50']),
        'dev_test_map50_all_classes': float(base_dev_test_metrics['map50']),
        'test_map50_all_classes': float(base_test_metrics['map50']),
        'val_ap50_person': float(base_val_person),
        'dev_test_ap50_person': float(base_dev_test_person),
        'test_ap50_person': float(base_test_person),
        'val_seconds_per_image': float(base_val_seconds / max(1, len(val_ds_eval))),
        'dev_test_seconds_per_image': float(base_dev_test_seconds / max(1, len(dev_test_ds_eval))),
        'test_seconds_per_image': float(base_test_seconds / max(1, len(test_ds_eval))),
    },
]).sort_values('val_map50_all_classes', ascending=False).reset_index(drop=True)

comparison_df['delta_vs_base_val_map50'] = comparison_df['val_map50_all_classes'] - float(base_val_metrics['map50'])
comparison_df['delta_vs_base_dev_test_map50'] = comparison_df['dev_test_map50_all_classes'] - float(base_dev_test_metrics['map50'])
comparison_df['delta_vs_base_person_ap50'] = comparison_df['val_ap50_person'] - float(base_val_person)
comparison_df.to_csv(comparison_path, index=False)

print('Best fine-tuned experiment:', best_experiment)
print('Base model threshold (val tuned):', round(best_base_threshold, 2))
print('Saved base val predictions:', base_val_path)
print('Saved base dev_test predictions:', base_dev_test_path)
print('Saved base test predictions:', base_test_path)
print('Saved base threshold tuning:', base_tuning_path)
print('Saved comparison table:', comparison_path)

display(base_threshold_df)
display(comparison_df)

# Optional cleanup for long sessions.
del base_model
if torch.cuda.is_available():
    torch.cuda.empty_cache()